# Query Generation, Routing, and Retrieval postprocessing

In [1]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import getpass

In [2]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [25]:
uk_with_metadata_collection = Chroma(
    collection_name="uk_with_metadata_collection",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY))
uk_with_metadata_collection.reset_collection()

In [26]:
# DEFINING THE INGESTION CONTENT AND SPLITTING STRATEGY

from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [27]:
html2text_transformer = Html2TextTransformer()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [28]:
def split_docs_into_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    chunks = text_splitter.split_documents(text_docs)
    return chunks

In [29]:
uk_destinations = [
    ("Cornwall", "Cornwall"), ("North_Cornwall", "Cornwall"),
    ("South_Cornwall", "Cornwall"), ("West_Cornwall", "Cornwall"),
    ("Tintagel", "Cornwall"), ("Bodmin", "Cornwall"),
    ("Wadebridge", "Cornwall"),
    ("Penzance", "Cornwall"), ("Newquay", "Cornwall"),
    ("St_Ives", "Cornwall"),
    ("Port_Isaac", "Cornwall"), ("Looe", "Cornwall"),
    ("Polperro", "Cornwall"),
    ("Porthleven", "Cornwall"),
    ("East_Sussex", "East_Sussex"), ("Brighton", "East_Sussex"),
    ("Battle", "East_Sussex"), ("Hastings_(England)", "East_Sussex"),
    ("Rye_(England)", "East_Sussex"), ("Seaford", "East_Sussex"),
    ("Ashdown_Forest", "East_Sussex")
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

uk_destination_url_with_metadata = [
    ( f'{wikivoyage_root_url}/{destination}', destination, region)
    for destination, region in uk_destinations]

In [30]:
# INGESTING CONTENT WITH METADATA
for (url, destination, region) in uk_destination_url_with_metadata:
    html_loader = AsyncHtmlLoader(url)
    docs = html_loader.load()
    
    docs_with_metadata = [
        Document(page_content=d.page_content,
        metadata = {
            'source': url,
            'destination': destination,
            'region': region})
        for d in docs]
    
    chunks = split_docs_into_chunks(docs_with_metadata)
    
    print(f'Importing: {destination}')
    uk_with_metadata_collection.add_documents(documents=chunks)

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  4.66it/s]


Importing: Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.92it/s]


Importing: North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.07it/s]


...

Importing: Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.45it/s]


Importing: Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  6.72it/s]


Importing: Ashdown_Forest


### Q&A on a metadata-enriched collection

In [31]:
# QUERYING WITH AN EXPLICIT METADATA FILTER

question = "Events or festivals"
metadata_retriever = uk_with_metadata_collection.as_retriever(search_kwargs={'k':2, 'filter':{'destination': 'Newquay'}})
result_docs = metadata_retriever.invoke(question)
print(result_docs)

[Document(id='15618677-e20b-412c-bf83-f5872f214cda', metadata={'destination': 'Newquay', 'source': 'https://en.wikivoyage.org/wiki/Newquay', 'region': 'Cornwall'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay.  (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members.  (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"), Document(id='1a37e113-954d-43b7-8eaf-89cddf4682b2', metadata={'destination': 'Newquay', 'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content="### Mid-range\n\n[edit]\n\n  * 50.41326-5.0855229 Concho Lounge, 16 Bank St. £10-25.  (updated Feb 2023)\n  * 50.4141

In [32]:
# AUTOMATICALLY GENERATING METADATA FILTERS WITH SELFQUERYRETRIEVER

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import ChatOpenAI

In [33]:
metadata_field_info = [
    AttributeInfo(
        name="destination",
        description="The specific UK destination to be searched",
        type="string",
    ),
    AttributeInfo(
        name="region",
        description="The name of the UK region to be searched",
        type="string",
    )
]

question = "Tell me about events or festivals in the UK town of Newquay"

llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm, uk_with_metadata_collection, question,
    metadata_field_info, verbose=True
)

result_docs = self_query_retriever.invoke(question)
print(result_docs)

[Document(id='15618677-e20b-412c-bf83-f5872f214cda', metadata={'destination': 'Newquay', 'source': 'https://en.wikivoyage.org/wiki/Newquay', 'region': 'Cornwall'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay.  (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members.  (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"), Document(id='a4d380d9-bd1a-4449-b53d-d66025b9332c', metadata={'destination': 'Newquay', 'source': 'https://en.wikivoyage.org/wiki/Newquay', 'region': 'Cornwall'}, page_content="## Drink\n\n[edit]\n\nNewquay's town centre is home to a large number of pubs and bars.\n\n  * **The Central Inn** \\

In [34]:
# GENERATING METADATA FILTERS WITH AN LLM FUNCTION CALL

import datetime
from typing import Literal, Optional, Tuple, List
from pydantic import BaseModel, Field
from langchain_classic.chains.query_constructor.ir import(
    Comparator,
    Comparison,
    Operation,
    Operator,
    StructuredQuery,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator


In [35]:
class DestinationSearch(BaseModel):
    """Search over a vector database of tourist destinations."""

    content_search: str = Field(
        "",
        description="""Similarity search query applied to tourist destinations.""",
    )
    
    destination: str = Field(
        ...,
        description="The name of the UK region to be searched.",
    )

    region: str = Field(
        ...,
        description="The name of the UK region to be searched.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(self, field) != getattr(self.__fields__[field], "default", None):
                print(f"{field}: {getattr(self, field)}")

In [36]:
# BUILDING A CHROMADB FILTER STATEMENT FROM THE STRUCTURED QUERY

def build_filter(destination_search: DestinationSearch):
    comparisons = []

    destination = destination_search.destination
    region = destination_search.region

    if destination and destination != '':
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="destination",
                value=destination,
            )
        )

    if region and region != '':
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="region",
                value=region
            )
        )

    search_filter = Operation(operator=Operator.AND, arguments=comparisons)

    chroma_filter = ChromaTranslator().visit_operation(search_filter)

    return chroma_filter

In [37]:
# BUILDING A QUERY CHAIN TO CONVERT THE QUESTION INTO A STRUCTURED QUERY

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system_message = """You are an expert at converting user
questions into vector database queries.
You have access to a database of tourist destinations.
Given a question, return a database query optimized
to retrieve the most relevant results.
If there are acronyms or words you are not familiar with,
do not try to rephrase them."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", "{question}"),
    ]
)

llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
structured_llm = llm.with_structured_output(DestinationSearch, method="function_calling")

query_generator = prompt | structured_llm

question = "Tell me about events or festivals in the UK town of Newquay"

structured_query = query_generator.invoke(question)
print(structured_query)

content_search='events festivals' destination='Newquay' region='Cornwall'


In [38]:
search_filter = build_filter(structured_query)
print(search_filter)

{'$and': [{'destination': {'$eq': 'Newquay'}}, {'region': {'$eq': 'Cornwall'}}]}


In [39]:
search_query = structured_query.content_search

metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':3, 'filter': search_filter})

answer = metadata_retriever.invoke(search_query)
print(answer)

[Document(id='15618677-e20b-412c-bf83-f5872f214cda', metadata={'source': 'https://en.wikivoyage.org/wiki/Newquay', 'region': 'Cornwall', 'destination': 'Newquay'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay.  (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members.  (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"), Document(id='af47969b-97c2-4edf-9d6c-a07ee130d121', metadata={'region': 'Cornwall', 'destination': 'Newquay', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content="## Eat\n\n[edit]\n\n### Budget\n\n[edit]\n\nThere are lots of cheap eats in the town centre.\n\n  * 50.415513-5.08688

## Generating a structured SQL query

In [40]:
from langchain_community.utilities import SQLDatabase
from langchain_community.tools import QuerySQLDataBaseTool
from langchain_classic.chains import create_sql_query_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

import getpass
import os

In [41]:
db = SQLDatabase.from_uri("sqlite:///UKBooking.db")
print(db.get_usable_table_names())

['Accommodation', 'AccommodationType', 'Booking', 'Customer', 'Destination', 'Offer']


In [42]:
db.run("select * from offer;")

"[(1, 1, 'Summer Special', 0.15, '2024-06-01', '2024-08-31'), (2, 2, 'Weekend Getaway', 0.1, '2024-09-01', '2024-12-31'), (3, 3, 'Early Bird Discount', 0.2, '2024-05-01', '2024-06-30'), (4, 4, 'Stay 3 Nights, Get 1 Free', 0.25, '2024-01-01', '2024-03-31'), (5, 5, 'Historic Stay Offer', 0.1, '2024-04-01', '2024-06-30'), (6, 6, 'Autumn Discount', 0.15, '2024-09-01', '2024-11-30'), (7, 7, 'Cottage Retreat Offer', 0.12, '2024-07-01', '2024-09-30'), (8, 8, 'City Break Deal', 0.08, '2024-10-01', '2024-12-31'), (9, 9, 'Luxury Villa Offer', 0.18, '2024-05-01', '2024-08-31'), (10, 10, 'Spa & Wellness Package', 0.2, '2024-04-01', '2024-07-31')]"

### Generating SQL queries from natural language

In [43]:
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4.1")
sql_query_gen_chain = create_sql_query_chain(llm, db)
response = sql_query_gen_chain.invoke(
{"question":
"Give me some offers for Cardiff, including the hotel name"})

print(response)

SQLQuery:
SELECT "Offer"."OfferDescription", "Offer"."DiscountRate", "Offer"."StartDate", "Offer"."EndDate", "Accommodation"."Name"
FROM "Offer"
JOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId"
JOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId"
WHERE "Destination"."Name" = 'Cardiff'
LIMIT 5;


In [44]:
db.run(response)

OperationalError: (sqlite3.OperationalError) near "SQLQuery": syntax error
[SQL: SQLQuery:
SELECT "Offer"."OfferDescription", "Offer"."DiscountRate", "Offer"."StartDate", "Offer"."EndDate", "Accommodation"."Name"
FROM "Offer"
JOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId"
JOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId"
WHERE "Destination"."Name" = 'Cardiff'
LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [45]:
clean_sql_prompt_template = """You are an expert in SQLite.
You are asked to fix badly formed SQLite queries,
which might contain unneeded prefixes or suffixes.
Given the following unclean SQL statement,
transform it to a clean,
executable SQL statement for SQLite.
Always prefix column names with the table name.
Only return an executable SQL statement which terminates
with a semicolon. Do not return anything else.
Do not include the language name or symbols like ```.

Unclean SQL: {unclean_sql}"""

clean_sql_prompt = ChatPromptTemplate.from_template(clean_sql_prompt_template)
clean_sql_chain = clean_sql_prompt | llm
full_sql_gen_chain = sql_query_gen_chain | clean_sql_chain | StrOutputParser()

In [46]:
question = """Give me some offers for Cardiff,
including the accommodation name"""
response = full_sql_gen_chain.invoke({"question": question})
print(response)

SELECT Offer.OfferDescription, Offer.DiscountRate, Accommodation.Name
FROM Offer
JOIN Accommodation ON Offer.AccommodationId = Accommodation.AccommodationId
JOIN Destination ON Accommodation.DestinationId = Destination.DestinationId
WHERE Destination.Name = 'Cardiff'
LIMIT 5;


In [47]:
db.run(response)

"[('Early Bird Discount', 0.2, 'Cardiff Camping')]"

### Executing the SQL query

In [48]:
sql_query_exec_chain = QuerySQLDataBaseTool(db=db)

sql_query_gen_and_exec_chain = full_sql_gen_chain | sql_query_exec_chain | StrOutputParser()

response = sql_query_gen_and_exec_chain.invoke({"question":question})
print(response)

[('Early Bird Discount', 0.2, 'Cardiff Camping')]


## Chain routing

### Setting up data retrievers

In [55]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableLambda

In [56]:
tourist_info_retriever_chain = RunnableLambda(
    lambda x: x['question']) \
        | uk_with_metadata_collection.as_retriever(
            search_kwargs={'k':2})

uk_accommodation_retriever_chain = full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

### Setting up the query router

In [57]:
class RouteQuery(BaseModel):
    """Route a user question to the most relevant datasource."""
    
    datasource: Literal["tourist_info_store",
        "uk_booking_db"] = Field(
        ...,
        description="""Given a user question,
            route it either to a tourist info vector store
            or a UK accommodation booking relational database.""",
    )

In [58]:
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-5-nano")
structured_llm_router = llm.with_structured_output(RouteQuery)

system = """You are an expert at routing a user question
to a tourist info vector store
or to an UK accommodation booking relational database.
The vector store contains tourist information about UK destinations.
Use the vector store for general tourist information questions
on UK destinations.
For questions about accommodation availability or booking,
use the UK Booking database."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

In [59]:
# TESTING THE ROUTER CHAIN
selected_data_source = question_router.invoke(
    {"question": "Have you got any offers in Brighton?"}
)
print(selected_data_source)

datasource='tourist_info_store'


In [60]:
selected_data_source = question_router.invoke(
    {"question": "Where are the best beaches in Cornwall?"}
)
print(selected_data_source)

datasource='tourist_info_store'


In [61]:
# SETTING UP THE RETRIEVER CHOOSER
retriever_chains = {
    'tourist_info_store': tourist_info_retriever_chain,
    'uk_booking_db': uk_accommodation_retriever_chain
}

def retriever_chooser(question):
    selected_data_source = question_router.invoke(
        {"question": question})
    return retriever_chains[selected_data_source.datasource]


In [62]:
chosen = retriever_chooser("""Tell me about events
    or festivals in the UK town of Newquay""")
print(chosen)

first=RunnableLambda(...) middle=[] last=VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002A4FEB9A490>, search_kwargs={'k': 2})


### Integrating the chain router into a full RAG chain

In [66]:
from langchain_core.runnables import RunnablePassthrough

rag_prompt_template = """
Given a question and some context, answer the question.
If you get a structured context, like a tuple, try to
infer the meaning of the components:
typically they refer to accommodation offers,
and the number is a percentage (0.2 means 20%).
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

In [67]:
def execute_rag_chain(question, chosen_retriever):
    full_rag_chain = (
    {
        "context": {"question": RunnablePassthrough()}
            | chosen_retriever,
            "question": RunnablePassthrough(),
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    return full_rag_chain.invoke(question)

In [69]:
#EXAMPLE: ASKING ABOUT ACCOMMODATION OFFERS
question = """Give me some offers for Cardiff,
including the accommodation name"""

chosen_retriever = retriever_chooser(question)
answer = execute_rag_chain(question, chosen_retriever)
print(answer)

- Cardiff Camping — Early Bird Discount: 20% off your stay.


In [70]:
# EXAMPLE: ASKING ABOUT TOURIST INFORMATION

question_2 = """Tell me about events or festivals
in the UK town of Newquay"""
chosen_retriever_2 = retriever_chooser(question_2)
answer2 = execute_rag_chain(question_2, chosen_retriever_2)
print(answer2)

- Cornish Film Festival: held annually around November, in the Newquay area.  
- Surfing-related events: Newquay is described as the UK’s surfing capital, with events such as the UK surfing championships and the Boardmasters festival (a surfing/music festival) taking place on the area’s beaches.


## Retrieval postprocessing

### RAG fusion (Reciprocal Rank Fusion)

In [71]:
from langchain_core.prompts import ChatPromptTemplate
from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

In [72]:
multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task is
to generate five different versions of the given user
question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the
user question, your goal is to help
the user overcome some of the limitations of the
distance-based similarity search.
Provide these alternative questions separated by newlines.
Original question: {question}
"""

In [73]:
multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template)

In [74]:
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""
    
    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))

questions_parser = LineListOutputParser()
llm = ChatOpenAI(model="gpt-5", openai_api_key=OPENAI_API_KEY)
multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

In [75]:
# RRF ALGORITHM

def reciprocal_rank_fusion(results_groups: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple groups of
    ranked documents and an optional parameter k used in
    the Reciprocal Rank Fusion (RRF) formula """
    
    indexed_results = {}
    
    for group_id, results_group in enumerate(results_groups):
        for local_rank, doc in enumerate(results_group):
            indexed_results[(group_id, local_rank)] = doc
    
    fused_scores = {}
    
    for key, doc in indexed_results.items():
        group_id, local_rank = key
        
        if key not in fused_scores:
            fused_scores[key] = 0
        
        doc_current_score = fused_scores[key]
        fused_scores[key] += 1 / (local_rank + k)
    
    reranked_results = [
        (indexed_results[key], score)
        for key, score in sorted(fused_scores.items(),
            key=lambda x: x[1], reverse=True)
    ]
    
    return reranked_results

In [76]:
# SETTING UP THE RAG FUSION RETRIEVAL CHAIN

retriever = uk_with_metadata_collection.as_retriever(search_kwargs={'k':3})

top_three_results = RunnableLambda(lambda x: x[0:3])

rag_fusion_retrieval_chain =multi_query_gen_chain \
    | retriever.map() | reciprocal_rank_fusion \
    | top_three_results

docs = rag_fusion_retrieval_chain.invoke(
    {"question": question})

len(docs)

3

In [78]:
# INCORPORATING RAG FUSION INTO THE RAG CHAIN

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.
Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} | rag_fusion_retrieval_chain,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

user_question = "Can you give me some tips for a trip to Brighton?"
answer = rag_chain.invoke(user_question)
print(answer)

Here are some quick, practical tips for a Brighton trip:

- When to go: Spring is lively, and May brings Brighton Festival and the Festival Fringe. Summer is ideal for lazy days and beautiful sunsets on the seafront.
- The beach: Expect a more than 5 mi (8 km) stretch of shingle (pebble) beach facing south onto the English Channel—great for sunset strolls. Plan footwear/seating accordingly since it’s not sandy.
- Getting in: Trains are a convenient way to arrive. (There’s a general guide to rail travel in Great Britain if you need it.)
- Getting around: You’ll find multiple options—bike, bus, train, and taxis—once in the city.
- Areas to explore for food and drinks: City Centre and elsewhere for all budgets; for nightlife and pubs, check out Station/Trafalgar Street, North Laine, The Lanes, the Pavilion area, Churchill Square and the Seafront, Western Road, Hove, St James’s Street/Kemptown, Hanover, and Northern Brighton. There are also clubs and live music venues.
- Events: If you’re 